# DINOv2 probe на инвентарных кропах

In [ ]:
!pip -q install -U timm

In [ ]:
import csv, json, zipfile, urllib.request
from pathlib import Path
import numpy as np, pandas as pd, torch, timm
from torchvision import transforms
from PIL import Image
from sklearn.linear_model import LogisticRegression

WORK=Path("/content")
url="https://raw.githubusercontent.com/Quadrotez/hackaton-cu-2026/crop-model/colab/inventory_crops.zip"
urllib.request.urlretrieve(url, WORK/"inv.zip")
with zipfile.ZipFile(WORK/"inv.zip") as z: z.extractall(WORK/"d")
D=WORK/"d"
lab=pd.read_csv(D/"train_labels.csv"); oof=pd.read_csv(D/"oof.csv"); test=pd.read_csv(D/"test.csv")
MATS=["metal","wood","plastic","glass","paper","mineral"]
print("device", "cuda" if torch.cuda.is_available() else "cpu", len(lab), len(oof), len(test))
assert torch.cuda.is_available(), "GPU T4 required"

In [ ]:
def softmax(x):
    e=np.exp(x-x.max(1,keepdims=True)); return e/e.sum(1,keepdims=True)
oof_p=np.zeros((len(oof),7)); test_b=np.zeros(len(test)); test_m=np.zeros((len(test),6))
pos={ (r.image_id,int(r.hyp_idx)):i for i,r in enumerate(lab.itertuples()) }
for fold in range(1,6):
    tr=lab[(lab.fold!=fold)&lab.material.isin(MATS)]
    va_idx=[pos[(r.image_id,int(r.hyp_idx))] for r in oof.itertuples() if int(r.fold)==fold and (r.image_id,int(r.hyp_idx)) in pos]
    Xtr=EF[tr.index.to_numpy()]; ytr=tr.form.to_numpy(); ym=tr.material.to_numpy()
    fb=LogisticRegression(C=1,class_weight='balanced',max_iter=2000).fit(Xtr,ytr)
    fm=LogisticRegression(C=1,class_weight='balanced',max_iter=2000).fit(Xtr,ym)
    for i in va_idx:
        oof_p[i,0]=fb.predict_proba(EF[i:i+1])[0,list(fb.classes_).index('bulk')]
        pr=fm.predict_proba(EF[i:i+1])[0]; oof_p[i,1:]=[pr[list(fm.classes_).index(x)] if x in fm.classes_ else 0 for x in MATS]
    test_b+=fb.predict_proba(ET)[:,list(fb.classes_).index('bulk')]/5
    pr=fm.predict_proba(ET); test_m+=np.column_stack([pr[:,list(fm.classes_).index(x)] if x in fm.classes_ else np.zeros(len(ET)) for x in MATS])/5
    print('fold',fold,flush=True)
with (WORK/'oof_probs.csv').open('w',newline='') as fh:
    w=csv.writer(fh); w.writerow(['image_id','hyp_idx','fold','detection_sha256','p_bulk']+[f'p_material_{x}' for x in MATS])
    for i,r in enumerate(oof.itertuples()): w.writerow([r.image_id,r.hyp_idx,r.fold,r.detection_sha256,f'{oof_p[i,0]:.8f}']+[f'{v:.8f}' for v in oof_p[i,1:]])
with (WORK/'test_probs.csv').open('w',newline='') as fh:
    w=csv.writer(fh); w.writerow(['image_id','hyp_idx','detection_sha256','p_bulk']+[f'p_material_{x}' for x in MATS])
    for i,r in enumerate(test.itertuples()): w.writerow([r.image_id,r.hyp_idx,r.detection_sha256,f'{test_b[i]:.8f}']+[f'{v:.8f}' for v in test_m[i]])
print('SAVED',flush=True)